# Phase 0 — Train / Validation Split

Splits **Source-1 entities** (and their ground-truth labels) into a train and
validation set, stratified by `country` and `match-count bucket`, so the
validation set mirrors the real distribution — including the singleton rate.

**Rule (do not break this):** only the S1 / ground-truth side gets split.
Source2 and Source3 are never split — every S1 entity (train or val) still
searches across the *full* S2/S3 pool, exactly like the real test set will.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_DIR = "../../../dataset/train"
SPLIT_DIR = "../../../dataset/split"
RANDOM_SEED = 42
VAL_FRACTION = 0.2

import os
os.makedirs(SPLIT_DIR, exist_ok=True)

## 1. Load ground truth + Source-1 country column

In [2]:
gt = pd.read_csv(
    f"{TRAIN_DIR}/train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

s1 = pd.read_csv(
    f"{TRAIN_DIR}/train_source1.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
    usecols=["entity_id", "country"],
)

print(f"Ground truth rows: {len(gt):,}")
print(f"Source1 rows:      {len(s1):,}")
gt.head()

Ground truth rows: 2,206,821
Source1 rows:      2,206,821


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


## 2. Build stratification key

In [3]:
gt["match_count"] = gt["matched_entity_ids"].apply(
    lambda x: 0 if x == "" else len(x.split(","))
)

gt["bucket"] = pd.cut(
    gt["match_count"],
    bins=[-1, 0, 1, 3, 5, 100],
    labels=["0", "1", "2-3", "4-5", "6+"],
)

gt = gt.merge(s1, left_on="source1_entity_id", right_on="entity_id", how="left")
gt["strata"] = gt["country"].astype(str) + "_" + gt["bucket"].astype(str)

print("Strata distribution (full set):")
gt["strata"].value_counts().sort_index()

Strata distribution (full set):


strata
India_0       49351
India_1       47468
India_2-3    361892
India_4-5    323008
India_6+     101469
US_0          73896
US_1          71689
US_2-3       544161
US_4-5       483064
US_6+        150823
Name: count, dtype: int64

## 3. Stratified 80/20 split at the S1 entity level

In [4]:
train_ids, val_ids = train_test_split(
    gt["source1_entity_id"],
    test_size=VAL_FRACTION,
    random_state=RANDOM_SEED,
    stratify=gt["strata"],
)

print(f"Train S1 entities: {len(train_ids):,}")
print(f"Val S1 entities:   {len(val_ids):,}")

assert set(train_ids).isdisjoint(set(val_ids)), "Train/val overlap detected!"
print("No overlap between train and val — confirmed.")

Train S1 entities: 1,765,456
Val S1 entities:   441,365


No overlap between train and val — confirmed.


## 4. Sanity check — strata proportions should closely match between train and val

In [5]:
gt_indexed = gt.set_index("source1_entity_id")
train_strata = gt_indexed.loc[train_ids, "strata"].value_counts(normalize=True)
val_strata = gt_indexed.loc[val_ids, "strata"].value_counts(normalize=True)

comparison = pd.DataFrame({"train_pct": train_strata, "val_pct": val_strata}).sort_index()
comparison

,train_pct,val_pct
strata,,
India_0,0.022363,0.022362
India_1,0.021509,0.021511
India_2-3,0.163988,0.163987
India_4-5,0.146368,0.146369
India_6+,0.045980,0.045980
US_0,0.033485,0.033485
US_1,0.032485,0.032486
US_2-3,0.246582,0.246580
US_4-5,0.218896,0.218896


## 5. Save split outputs

In [6]:
train_ids.to_csv(f"{SPLIT_DIR}/train_s1_ids.txt", index=False, header=False)
val_ids.to_csv(f"{SPLIT_DIR}/val_s1_ids.txt", index=False, header=False)

gt_train = gt[gt["source1_entity_id"].isin(set(train_ids))].drop(
    columns=["entity_id", "bucket", "strata", "match_count", "country"]
)
gt_val = gt[gt["source1_entity_id"].isin(set(val_ids))].drop(
    columns=["entity_id", "bucket", "strata", "match_count", "country"]
)

gt_train.to_csv(f"{SPLIT_DIR}/train_ground_truth.tsv", sep="\t", index=False)
gt_val.to_csv(f"{SPLIT_DIR}/val_ground_truth.tsv", sep="\t", index=False)

print("Saved:")
print(f"  {SPLIT_DIR}/train_s1_ids.txt")
print(f"  {SPLIT_DIR}/val_s1_ids.txt")
print(f"  {SPLIT_DIR}/train_ground_truth.tsv")
print(f"  {SPLIT_DIR}/val_ground_truth.tsv")

Saved:
  ../../../dataset/split/train_s1_ids.txt
  ../../../dataset/split/val_s1_ids.txt
  ../../../dataset/split/train_ground_truth.tsv
  ../../../dataset/split/val_ground_truth.tsv


## Next step

Phase 0 is complete. `val_ground_truth.tsv` is now set aside — never train
on it, only score against it later once you have predictions.

**Do not re-run this notebook with a different `RANDOM_SEED` mid-hackathon** —
the split must stay fixed so F_0.5 numbers stay comparable across every
experiment your team runs from here on.